In [ ]:
# IPython magig  tools
%load_ext autoreload
%autoreload 2

import subprocess

# Mount the drive from Python
# subprocess.run(['open', 'smb://allen/aind'])

from aind_vr_foraging_analysis.utils.parsing import data_access
import aind_vr_foraging_analysis.utils.plotting as plotting

# Plotting libraries
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.backends.backend_pdf import PdfPages

import matplotlib.patches as mpatches
# Plotting libraries
import seaborn as sns
from matplotlib.gridspec import GridSpec
from matplotlib.ticker import FixedLocator, FuncFormatter

import warnings
pd.options.mode.chained_assignment = None  # Ignore SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

color1='#d95f02'
color2='#1b9e77'
color3="#433abf"
color4='yellow'
odor_list_color = [color1, color2, color3, color4]
import matplotlib.lines as mlines

pdf_path = r'Z:\scratch\vr-foraging\sessions'
results_path = r'C:\Users\tiffany.ona\OneDrive - Allen Institute\Documents\VR foraging\experiments\batch 5 - learning\results'

color_dict_label = {'InterSite': '#808080',
            'InterPatch': '#b3b3b3', 'PatchZ': '#d95f02',
            'PatchZA': '#d95f02', 'PatchZB': '#d95f02', 
            'PatchB': '#d95f02','PatchA': '#7570b3', 
            'PatchC': '#1b9e77',
            'Alpha-pinene': '#1b9e77', 
            'Methyl Butyrate': '#7570b3', 
            'Amyl Acetate': '#d95f02', 
            'Fenchone': '#7570b3', 
            'Dipropyl sulfide': '#7570b3',
            'Hexanal': '#1b9e77',
            'Pentyl acetate': '#d95f02',
            'S': color1,
            'D': color2,
            'N': color3,   
            'Do': color1,
            'None': color4
            }

label_dict = {**{
"InterSite": '#808080',
"InterPatch": '#b3b3b3'}, 
            **color_dict_label}
import os
import re
sns.set_context('talk')

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
import inspect
from aind_vr_foraging_analysis.utils.parsing.data_access import load_session
print(inspect.getsource(load_session))

def load_session(session_path: Path):
    """
    Loads and processes a behavioral session from a given path.
    
    Parameters:
    ----------
    session_path : Path
        Full path to the session directory containing the raw data.

    Returns:
    -------
    all_epochs : pd.DataFrame
        A DataFrame of parsed and enriched behavioral epochs from the session.
        
    stream_data : object
        An object containing continuous data streams (e.g., analog signals, encoder).
        
    data : dict or object
        The raw session data as returned by `parse.load_session_data()`.
    """
    
    data = parse.load_session_data(session_path)

    all_epochs = parse.parse_dataframe(data)

    extra_columns = AddExtraColumns(all_epochs, run_on_init=True)
    all_epochs = extra_columns.get_all_epochs()

    stream_data = parse.ContinuousData(data)
    
    return all_epochs, stream_data, data



In [41]:
date_string = "2024-4-1"
mouse_list = [
    
            '754570','754579','754567','754580','754559','754560','754577',
              '754566','754571','754572','754573','754574','754575', 
              '754582','745302','745305','745301',
              
              "715866", "713578", "707349", "716455", 
              "716458","715865", "715869","713545","715867",
              "715870","694569", 
              
              '789914', '789915', '789923', '789917', 
               '789913', '789909', '789910', '789911', '789921', 
               '789918', '789919', '789907', '789903', '789925', 
               '789924', '789926', '789908', '788641', '781898', '781896']

experiment_list = { 1: 'pilot',
                    2 : 'volume_manipulation',
                    3: 'global_reward_rate_patches', 
                    4: 'global_reward_rate_distance_friction',
                    5 : 'learning_reversals'}

In [39]:
import shutil
from pathlib import Path

def download_complete_sessions(mouse, n_sessions=5, 
                               source_base='/Volumes/aind/scratch/vr-foraging/data',
                               dest_base='~/Downloads/vr_foraging_data',
                               date_string='2024-4-1'):
    """
    Copy entire session directories - no filtering
    """
    from aind_vr_foraging_analysis.utils.parsing.data_access import find_sessions_relative_to_date
    
    # Expand home directory
    dest_base = Path(dest_base).expanduser()
    dest_mouse_dir = dest_base / mouse
    dest_mouse_dir.mkdir(parents=True, exist_ok=True)
    
    # Find sessions
    session_paths = find_sessions_relative_to_date(
        mouse=mouse,
        date_string=date_string,
        when='on_or_after',
        base_path=source_base
    )
    
    # Take most recent n_sessions
    recent_sessions = session_paths[-n_sessions:] if len(session_paths) > n_sessions else session_paths
    
    print(f"Downloading {len(recent_sessions)} COMPLETE sessions for mouse {mouse}")
    print(f"Destination: {dest_mouse_dir}\n")
    
    for i, session_path in enumerate(recent_sessions, 1):
        session_name = session_path.name
        dest_session = dest_mouse_dir / session_name
        
        print(f"[{i}/{len(recent_sessions)}] Copying entire session {session_name}...")
        
        try:
            # Copy the entire session directory
            shutil.copytree(session_path, dest_session, dirs_exist_ok=True)
            
            # Get size
            total_size = sum(f.stat().st_size for f in dest_session.rglob('*') if f.is_file())
            size_mb = total_size / (1024 * 1024)
            
            print(f"  ✓ Complete ({size_mb:.1f} MB)")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\n✓ Download complete! Data saved to: {dest_mouse_dir}")
    return dest_mouse_dir

# Download 5 complete sessions
local_path = download_complete_sessions('754560', n_sessions=5)
local_path = download_complete_sessions('754582', n_sessions=5)
local_path = download_complete_sessions('754577', n_sessions=5)
local_path = download_complete_sessions('745305', n_sessions=5)
local_path = download_complete_sessions('745301', n_sessions=5)

Destination: /Users/laura.driscoll/Downloads/vr_foraging_data/754560

[1/5] Copying entire session 754560_20241212T091529...
  ✗ Error: [('/Volumes/aind/scratch/vr-foraging/data/754560/754560_20241212T091529/behavior/Logs/folder_rearranged_info.json', '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241212T091529/behavior/Logs/folder_rearranged_info.json', "[Errno 1] Operation not permitted: '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241212T091529/behavior/Logs/folder_rearranged_info.json'"), ('/Volumes/aind/scratch/vr-foraging/data/754560/754560_20241212T091529/behavior/Logs/robocopy.log', '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241212T091529/behavior/Logs/robocopy.log', "[Errno 1] Operation not permitted: '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241212T091529/behavior/Logs/robocopy.log'"), ('/Volumes/aind/scratch/vr-foraging/data/754560/754560_20241212T091529/rig.json', '/Users/laura.driscoll/D

OSError: [Errno 28] No space left on device: '/Users/laura.driscoll/Downloads/vr_foraging_data/745301'

In [23]:
def extract_window_data(sites_df, use_time=False):
    """
    Extract window_data from sites DataFrame
    
    Parameters:
    - sites_df: DataFrame from load_session
    - use_time: If True, use time_since_entry, else use distance since patch entry
    
    Returns:
    - window_data: array of shape (n_sites, 3)
      Each row: [patch_time_or_distance, reward, stopped]
    """
    
    # Filter for OdorSites only
    odor_sites = sites_df[sites_df['label'] == 'OdorSite'].copy()
    
    if len(odor_sites) == 0:
        return np.array([])
    
    window_data = []
    
    # Get patch entry positions for each patch
    patch_entries = {}
    for patch_num in odor_sites['patch_number'].unique():
        if pd.isna(patch_num):
            continue
        # Find first odor site in this patch
        patch_sites = sites_df[sites_df['patch_number'] == patch_num]
        first_odor = patch_sites[patch_sites['label'] == 'OdorSite'].iloc[0]
        # Patch entry is the start of the first odor site
        patch_entries[patch_num] = first_odor['start_position']
    
    for _, site in odor_sites.iterrows():
        patch_num = site['patch_number']
        
        # Use either time or distance since patch entry
        if use_time:
            patch_metric = site['time_since_entry'] if pd.notna(site['time_since_entry']) else 0.0
        else:
            # Distance since patch entry
            if pd.notna(patch_num) and patch_num in patch_entries:
                patch_metric = site['start_position'] - patch_entries[patch_num]
            else:
                patch_metric = 0.0
        
        reward = 1 if site['is_reward'] else 0
        stopped = 1 if site['is_choice'] else 0
        
        window_data.append([patch_metric, reward, stopped])
    
    return np.array(window_data)

# Test both modes
window_data_distance = extract_window_data(sites_df, use_time=False)
window_data_time = extract_window_data(sites_df, use_time=True)

print("Distance-based window_data (distance since patch entry):")
print(f"  Shape: {window_data_distance.shape}")
print(f"  First 10 rows:\n{window_data_distance[:10]}")
print(f"  Total rewards: {int(window_data_distance[:, 1].sum())}")
print(f"  Total stops: {int(window_data_distance[:, 2].sum())}")

print("\n\nTime-based window_data (time since patch entry):")
print(f"  Shape: {window_data_time.shape}")
print(f"  First 10 rows:\n{window_data_time[:10]}")
print(f"  Total rewards: {int(window_data_time[:, 1].sum())}")
print(f"  Total stops: {int(window_data_time[:, 2].sum())}")

Distance-based window_data (distance since patch entry):
  Shape: (22, 3)
  First 10 rows:
[[  0.           1.           1.        ]
 [ 37.18266283   1.           1.        ]
 [ 79.19925857   1.           1.        ]
 [117.30563684   1.           1.        ]
 [157.80573667   0.           1.        ]
 [197.45474222   1.           1.        ]
 [229.70950902   1.           1.        ]
 [266.96184302   1.           1.        ]
 [311.81644018   1.           1.        ]
 [355.31409648   1.           1.        ]]
  Total rewards: 20
  Total stops: 22


Time-based window_data (time since patch entry):
  Shape: (22, 3)
  First 10 rows:
[[ 57.508992   1.         1.      ]
 [ 69.056      1.         1.      ]
 [118.780992   1.         1.      ]
 [126.388992   1.         1.      ]
 [139.013984   0.         1.      ]
 [253.837984   1.         1.      ]
 [281.375008   1.         1.      ]
 [286.117984   1.         1.      ]
 [331.263008   1.         1.      ]
 [374.472      1.         1.      ]]
  To

In [24]:
def load_mouse_sessions(mouse, date_string, use_time=False, 
                        base_path='/Volumes/aind/scratch/vr-foraging/data'):
    """
    Load all sessions for a mouse and extract window_data
    
    Parameters:
    - mouse: Mouse ID string
    - date_string: Date string for filtering sessions
    - use_time: If True, use time since patch entry, else use distance
    - base_path: Path to data directory
    
    Returns:
    - list of window_data arrays, one per session
    """
    from aind_vr_foraging_analysis.utils.parsing.data_access import (
        find_sessions_relative_to_date, 
        load_session
    )
    
    # Find sessions
    session_paths = find_sessions_relative_to_date(
        mouse=mouse,
        date_string=date_string,
        when='on_or_after',
        base_path=base_path
    )
    
    print(f"Loading {len(session_paths)} sessions for mouse {mouse}")
    
    all_window_data = []
    
    for session_path in session_paths:
        try:
            # Load session
            sites_df, continuous_data, raw_data = load_session(session_path)
            
            # Extract window data
            window_data = extract_window_data(sites_df, use_time=use_time)
            
            if len(window_data) > 0:
                all_window_data.append(window_data)
                print(f"  Session {session_path.name}: {len(window_data)} sites, "
                      f"{int(window_data[:, 1].sum())} rewards")
            else:
                print(f"  Session {session_path.name}: No odor sites found")
                
        except Exception as e:
            print(f"  Error loading {session_path.name}: {e}")
            
    return all_window_data

# Test on one mouse
mouse = mouse_list[0]
all_sessions = load_mouse_sessions(mouse, date_string, use_time=False)

print(f"\n\nLoaded {len(all_sessions)} sessions")
print(f"First session shape: {all_sessions[0].shape}")
print(f"Total sites across all sessions: {sum(len(s) for s in all_sessions)}")

Loading 79 sessions for mouse 754570
Reward functions from software events
  Session 754570_20240903T122712: 22 sites, 20 rewards
Reward functions from software events
  Session 754570_20240904T114053: 5 sites, 5 rewards
Reward functions from software events
  Session 754570_20240905T112958: 9 sites, 7 rewards
Reward functions from software events
  Session 754570_20240906T105852: 175 sites, 107 rewards


KeyboardInterrupt: 

In [25]:
def load_recent_mice_sessions(n_mice=20, n_sessions=20, date_string='2024-4-1', 
                               use_time=False,
                               base_path='/Volumes/aind/scratch/vr-foraging/data'):
    """
    Load the most recent sessions for the most recent mice
    
    Parameters:
    - n_mice: Number of recent mice to load
    - n_sessions: Number of recent sessions per mouse
    - date_string: Date string for filtering sessions
    - use_time: If True, use time since patch entry, else use distance
    - base_path: Path to data directory
    
    Returns:
    - dict mapping mouse_id -> list of window_data arrays
    """
    from aind_vr_foraging_analysis.utils.parsing.data_access import (
        find_sessions_relative_to_date, 
        load_session
    )
    import os
    
    # Get all mice directories
    all_mice = sorted([d for d in os.listdir(base_path) 
                       if os.path.isdir(os.path.join(base_path, d)) 
                       and not d.startswith('.')],
                      reverse=True)  # Most recent first (assuming higher IDs = newer)
    
    # Take the most recent n_mice
    recent_mice = all_mice[:n_mice]
    print(f"Loading data for {len(recent_mice)} most recent mice: {recent_mice}")
    
    all_data = {}
    
    for mouse in recent_mice:
        print(f"\nProcessing mouse {mouse}...")
        
        try:
            # Find sessions
            session_paths = find_sessions_relative_to_date(
                mouse=mouse,
                date_string=date_string,
                when='on_or_after',
                base_path=base_path
            )
            
            # Take the most recent n_sessions
            recent_sessions = session_paths[-n_sessions:] if len(session_paths) > n_sessions else session_paths
            
            print(f"  Loading {len(recent_sessions)} recent sessions (out of {len(session_paths)} total)")
            
            mouse_data = []
            
            for session_path in recent_sessions:
                try:
                    # Load session
                    sites_df, continuous_data, raw_data = load_session(session_path)
                    
                    # Extract window data
                    window_data = extract_window_data(sites_df, use_time=use_time)
                    
                    if len(window_data) > 0:
                        mouse_data.append(window_data)
                    
                except Exception as e:
                    print(f"    Error loading {session_path.name}: {e}")
            
            if mouse_data:
                all_data[mouse] = mouse_data
                total_sites = sum(len(s) for s in mouse_data)
                total_rewards = sum(s[:, 1].sum() for s in mouse_data)
                print(f"  Loaded {len(mouse_data)} sessions, {total_sites} sites, {int(total_rewards)} rewards")
            else:
                print(f"  No valid sessions found")
                
        except Exception as e:
            print(f"  Error processing mouse {mouse}: {e}")
    
    return all_data

# Load the data
mouse_data = load_recent_mice_sessions(n_mice=20, n_sessions=20, use_time=False)

print(f"\n\n=== SUMMARY ===")
print(f"Total mice loaded: {len(mouse_data)}")
for mouse, sessions in mouse_data.items():
    print(f"  {mouse}: {len(sessions)} sessions")

Loading data for 20 most recent mice: ['test', 'TestMouse', 'FIP_test', '815104', '815103', '815102', '808729', '808728', '808619', '807093', '807086', '806527', '798279', '795556', '795133', '794591', '789926', '789925', '789924', '789923']

Processing mouse test...
No sessions found for mouse test with condition 'on_or_after' on 2024-4-1
  Loading 0 recent sessions (out of 0 total)
  No valid sessions found

Processing mouse TestMouse...
  Loading 1 recent sessions (out of 1 total)
No reward sites found
    Error loading TestMouse_2025-11-07T185917Z: 'is_reward'
  No valid sessions found

Processing mouse FIP_test...
  Error processing mouse FIP_test: '>=' not supported between instances of 'str' and 'datetime.date'

Processing mouse 815104...
  Loading 4 recent sessions (out of 4 total)
No reward sites found
    Error loading 815104_2025-09-19T175433Z: 'is_reward'
Reward functions from software events
Reward functions from software events
Reward functions from software events
  Load

KeyboardInterrupt: 

In [37]:
import shutil
from pathlib import Path

def download_mouse_sessions_full(mouse, n_sessions=20, 
                                  source_base='/Volumes/aind/scratch/vr-foraging/data',
                                  dest_base='~/Downloads/vr_foraging_data',
                                  date_string='2024-4-1'):
    """
    Copy all necessary files for load_session to work
    """
    from aind_vr_foraging_analysis.utils.parsing.data_access import find_sessions_relative_to_date
    
    # Expand home directory
    dest_base = Path(dest_base).expanduser()
    dest_mouse_dir = dest_base / mouse
    
    # Find sessions
    session_paths = find_sessions_relative_to_date(
        mouse=mouse,
        date_string=date_string,
        when='on_or_after',
        base_path=source_base
    )
    
    # Take most recent n_sessions
    recent_sessions = session_paths[-n_sessions:] if len(session_paths) > n_sessions else session_paths
    
    print(f"Downloading {len(recent_sessions)} complete sessions for mouse {mouse}")
    print(f"Destination: {dest_mouse_dir}\n")
    
    required_paths = [
        'behavior/SoftwareEvents',
        'behavior/Renderer',
        'behavior/Treadmill.harp',
        'behavior/UpdaterEvents',
        'behavior/Logs',  # ADD THIS - contains config files
        'session.json',
        'rig.json'
    ]
    
    for i, session_path in enumerate(recent_sessions, 1):
        session_name = session_path.name
        dest_session = dest_mouse_dir / session_name
        
        print(f"[{i}/{len(recent_sessions)}] Copying {session_name}...")
        
        try:
            for rel_path in required_paths:
                src_path = session_path / rel_path
                dest_path = dest_session / rel_path
                
                if src_path.exists():
                    if src_path.is_dir():
                        shutil.copytree(src_path, dest_path, dirs_exist_ok=True)
                    else:
                        dest_path.parent.mkdir(parents=True, exist_ok=True)
                        shutil.copy2(src_path, dest_path)
            
            print(f"  ✓ Complete")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\n✓ Download complete! Data saved to: {dest_mouse_dir}")
    return dest_mouse_dir

# Re-download with Logs folder included
print("Re-downloading with config files (Logs folder)...\n")
local_path = download_mouse_sessions_full('754560', n_sessions=20)

Re-downloading with config files (Logs folder)...

Destination: /Users/laura.driscoll/Downloads/vr_foraging_data/754560

[1/20] Copying 754560_20241123T104127...
  ✗ Error: [('/Volumes/aind/scratch/vr-foraging/data/754560/754560_20241123T104127/behavior/Logs/folder_rearranged_info.json', '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241123T104127/behavior/Logs/folder_rearranged_info.json', "[Errno 1] Operation not permitted: '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241123T104127/behavior/Logs/folder_rearranged_info.json'"), ('/Volumes/aind/scratch/vr-foraging/data/754560/754560_20241123T104127/behavior/Logs/robocopy.log', '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241123T104127/behavior/Logs/robocopy.log', "[Errno 1] Operation not permitted: '/Users/laura.driscoll/Downloads/vr_foraging_data/754560/754560_20241123T104127/behavior/Logs/robocopy.log'")]
[2/20] Copying 754560_20241125T093629...
  ✗ Error: [('/Volumes/aind/

In [38]:
from pathlib import Path

# Set the local path
local_base = str(Path('~/Downloads/vr_foraging_data').expanduser())

# Test loading from downloaded files
print("Testing load_session on downloaded data...\n")

try:
    from aind_vr_foraging_analysis.utils.parsing.data_access import load_session
    
    # Get the downloaded session paths
    local_mouse_dir = Path(local_base) / '754560'
    local_sessions = sorted(list(local_mouse_dir.glob('754560_*')))
    
    print(f"Found {len(local_sessions)} downloaded sessions")
    
    if local_sessions:
        # Test on first session
        test_session = local_sessions[0]
        print(f"\nTesting session: {test_session.name}")
        
        # Try to load
        sites_df, continuous_data, raw_data = load_session(test_session)
        
        print(f"✓ Session loaded successfully")
        print(f"  Sites shape: {sites_df.shape}")
        
        # Extract window_data
        window_data = extract_window_data(sites_df, use_time=False)
        
        print(f"✓ Window data extracted successfully")
        print(f"  Shape: {window_data.shape}")
        print(f"  First 5 rows:\n{window_data[:5]}")
        print(f"  Total rewards: {int(window_data[:, 1].sum())}")
        print(f"  Total stops: {int(window_data[:, 2].sum())}")
        
        # Test all sessions
        print(f"\n\nTesting all {len(local_sessions)} sessions...")
        success_count = 0
        for session_path in local_sessions:
            try:
                sites_df, _, _ = load_session(session_path)
                window_data = extract_window_data(sites_df, use_time=False)
                if len(window_data) > 0:
                    success_count += 1
            except Exception as e:
                print(f"  ✗ {session_path.name}: {e}")
        
        print(f"\n✓ Successfully extracted window_data from {success_count}/{len(local_sessions)} sessions")
        
        if success_count == len(local_sessions):
            print("\n🎉 All sessions working! You can disconnect from the network.")
        
    else:
        print("No sessions found in downloaded directory")
        
except Exception as e:
    print(f"✗ Error: {e}")
    import traceback
    traceback.print_exc()

Testing load_session on downloaded data...

Found 20 downloaded sessions

Testing session: 754560_20241123T104127
Reward functions from software events
✓ Session loaded successfully
  Sites shape: (1098, 32)
✓ Window data extracted successfully
  Shape: (489, 3)
  First 5 rows:
[[  0.           0.           0.        ]
 [  0.           1.           1.        ]
 [ 71.79051519   1.           1.        ]
 [145.59415443   1.           1.        ]
 [220.18518568   1.           1.        ]]
  Total rewards: 208
  Total stops: 449


Testing all 20 sessions...
Reward functions from software events
Reward functions from software events
Reward functions from software events
  ✗ 754560_20241127T095951: 'harp_behavior'
  ✗ 754560_20241129T102343: 'harp_behavior'
  ✗ 754560_20241130T103951: 'harp_behavior'
  ✗ 754560_20241202T101932: 'harp_behavior'
  ✗ 754560_20241203T094544: 'harp_behavior'
  ✗ 754560_20241204T091627: 'harp_behavior'
  ✗ 754560_20241205T094035: 'harp_behavior'
  ✗ 754560_20241206

In [36]:
from pathlib import Path

local_mouse_dir = Path('~/Downloads/vr_foraging_data/754560').expanduser()
sessions = sorted(list(local_mouse_dir.glob('754560_*')))

# Compare a working vs failing session
working = sessions[0]  # 754560_20241123T104127
failing = sessions[3]  # 754560_20241127T095951

print("Files in working session:")
working_files = set([str(p.relative_to(working)) for p in working.rglob('*') if p.is_file()])
for f in sorted(working_files):
    print(f"  {f}")

print("\n\nFiles in failing session:")
failing_files = set([str(p.relative_to(failing)) for p in failing.rglob('*') if p.is_file()])
for f in sorted(failing_files):
    print(f"  {f}")

print("\n\nFiles in working but NOT in failing:")
for f in sorted(working_files - failing_files):
    print(f"  {f}")

Files in working session:
  .DS_Store
  behavior/Behavior.harp/Behavior_0.bin
  behavior/Behavior.harp/Behavior_1.bin
  behavior/Behavior.harp/Behavior_10.bin
  behavior/Behavior.harp/Behavior_100.bin
  behavior/Behavior.harp/Behavior_101.bin
  behavior/Behavior.harp/Behavior_102.bin
  behavior/Behavior.harp/Behavior_103.bin
  behavior/Behavior.harp/Behavior_104.bin
  behavior/Behavior.harp/Behavior_105.bin
  behavior/Behavior.harp/Behavior_106.bin
  behavior/Behavior.harp/Behavior_107.bin
  behavior/Behavior.harp/Behavior_108.bin
  behavior/Behavior.harp/Behavior_109.bin
  behavior/Behavior.harp/Behavior_11.bin
  behavior/Behavior.harp/Behavior_110.bin
  behavior/Behavior.harp/Behavior_111.bin
  behavior/Behavior.harp/Behavior_112.bin
  behavior/Behavior.harp/Behavior_113.bin
  behavior/Behavior.harp/Behavior_114.bin
  behavior/Behavior.harp/Behavior_115.bin
  behavior/Behavior.harp/Behavior_116.bin
  behavior/Behavior.harp/Behavior_117.bin
  behavior/Behavior.harp/Behavior_118.bin
  

In [31]:
def load_session_minimal(session_path):
    """
    Load only what's needed for window_data extraction
    Returns just the sites_df without continuous data
    """
    from aind_vr_foraging_analysis.utils.parsing import parse
    import json
    
    session_path = Path(session_path)
    
    # Load software events
    software_events_path = session_path / 'behavior' / 'SoftwareEvents'
    
    def load_jsonl(filename):
        data = []
        filepath = software_events_path / filename
        if filepath.exists():
            with open(filepath, 'r') as f:
                for line in f:
                    if line.strip():
                        data.append(json.loads(line))
        return data
    
    # Load the events we need
    active_sites = load_jsonl('ActiveSite.json')
    give_reward = load_jsonl('GiveReward.json')
    choice_feedback = load_jsonl('ChoiceFeedback.json')
    
    # Use the parse module to create sites_df
    # This is what load_session does internally
    all_epochs = parse.get_reward_epochs(active_sites, give_reward, choice_feedback)
    
    return all_epochs

# Test on all sessions with minimal loader
print("Testing minimal loader on all sessions...\n")

success_count = 0
all_window_data = []

for session_path in sessions:
    try:
        sites_df = load_session_minimal(session_path)
        window_data = extract_window_data(sites_df, use_time=False)
        
        if len(window_data) > 0:
            all_window_data.append(window_data)
            success_count += 1
            print(f"✓ {session_path.name}: {len(window_data)} sites, {int(window_data[:, 1].sum())} rewards")
    except Exception as e:
        print(f"✗ {session_path.name}: {e}")

print(f"\n✓ Successfully extracted window_data from {success_count}/{len(sessions)} sessions")

Testing minimal loader on all sessions...

✗ 754560_20241123T104127: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_20241125T093629: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_20241126T100429: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_20241127T095951: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_20241129T102343: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_20241130T103951: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_20241202T101932: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_20241203T094544: module 'aind_vr_foraging_analysis.utils.parsing.parse' has no attribute 'get_reward_epochs'
✗ 754560_2024

In [33]:
import inspect
from aind_vr_foraging_analysis.utils.parsing import parse

print("parse_dataframe signature:")
print(inspect.signature(parse.parse_dataframe))

print("\n\nSource code (first 2000 chars):")
print(inspect.getsource(parse.parse_dataframe)[:2000])

parse_dataframe signature:
(data: dict) -> pandas.core.frame.DataFrame


Source code (first 2000 chars):
def parse_dataframe(data: dict) -> pd.DataFrame:
    """
    Parse the data from the session and return the reward sites, active sites and encoder data

    Inputs:
    data: dict
        Data from the session

    Returns:
    all_epochs: pd.DataFrame
        DataFrame containing the  active sites

    """
    data["software_events"].streams.ActiveSite.load_from_file()
    active_site_temp = data["software_events"].streams.ActiveSite.data

    # Use json_normalize to create a new DataFrame from the 'data' column
    active_site = pd.json_normalize(active_site_temp["data"])
    active_site.index = active_site_temp.index

    # Add the postpatch label
    active_site["previous_epoch"] = active_site["label"].shift(-1)
    active_site["label"] = np.where(
        active_site["label"] == active_site["previous_epoch"], "PostPatch", active_site["label"]
    )
    active_site.drop(columns=

In [15]:
def session_df_to_window_data(session_df):
    """
    Convert session DataFrame to window_data format for SBI
    
    window_data shape: (n_sites, 3)
    Each row: [patch_time, reward, stopped]
    """
    window_data = []
    
    for idx, row in session_df.iterrows():
        # patch_time: we don't have exact dwell time, so use a placeholder
        # You might need to extract this from treadmill or position data
        patch_time = 1.0 if row['is_choice'] else 0.0  # placeholder
        
        reward = row['is_reward']
        stopped = row['is_choice']
        
        window_data.append([patch_time, reward, stopped])
    
    return np.array(window_data)

window_data = session_df_to_window_data(session_df)
print(f"\nWindow data shape: {window_data.shape}")
print(f"First 5 rows:\n{window_data[:5]}")


Window data shape: (22, 3)
First 5 rows:
[[1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]
 [1. 1. 1.]]


In [16]:
# Check what's in the Behavior.harp folder - this likely has position/velocity data
behavior_harp_path = os.path.join(behavior_path, 'Behavior.harp')

# Also check Treadmill data
treadmill_path = os.path.join(behavior_path, 'Treadmill.harp')

print("Checking for position/movement data...")

# The Renderer folder might also have position info
renderer_path = os.path.join(behavior_path, 'Renderer')
if os.path.exists(renderer_path):
    print("\nRenderer folder contents:")
    for item in os.listdir(renderer_path)[:10]:
        print(f"  {item}")

# Look for CSV files that might have already-processed position data
print("\n\nLooking for processed data files in session root:")
for item in os.listdir(session_path):
    if item.endswith('.csv') or item.endswith('.parquet'):
        print(f"  {item}")

Checking for position/movement data...

Renderer folder contents:
  RendererSynchState.csv


Looking for processed data files in session root:


In [19]:
# Try the load_session function
from aind_vr_foraging_analysis.utils.parsing.data_access import load_session

# Load the session
session_data = load_session(session_path)

print("Session data type:", type(session_data))
print("\nSession data keys/attributes:")
if isinstance(session_data, dict):
    for key in session_data.keys():
        print(f"  {key}: {type(session_data[key])}")
elif hasattr(session_data, '__dict__'):
    for key in session_data.__dict__.keys():
        print(f"  {key}")
else:
    print(dir(session_data))

Reward functions from software events
Session data type: <class 'tuple'>

Session data keys/attributes:
['__add__', '__class__', '__class_getitem__', '__contains__', '__delattr__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__mul__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__rmul__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', 'count', 'index']


In [21]:
# Unpack the session data
sites_df, continuous_data, raw_data = session_data

print("Sites DataFrame shape:", sites_df.shape)
print("\nColumns we care about:")
print(sites_df[['label', 'start_position', 'length', 'patch_number', 
                'is_choice', 'is_reward', 'time_since_entry', 'stop_time']].head(10))

# Filter for OdorSites only (these are the reward sites)
odor_sites = sites_df[sites_df['label'] == 'OdorSite'].copy()
print(f"\n\nOdorSites only: {len(odor_sites)} sites")
print(odor_sites[['patch_number', 'start_position', 'time_since_entry', 
                   'is_choice', 'is_reward']].head(10))

Sites DataFrame shape: (46, 32)

Columns we care about:
                    label  start_position     length  patch_number is_choice  \
start_time                                                                     
241479.950976  InterPatch       -0.166820  58.843044             0       NaN   
241502.492992   InterSite       58.676223  21.652379             0       NaN   
241560.001984    OdorSite       80.328603  20.000000             0      True   
241565.512000   InterSite      100.328603  17.182663             0       NaN   
241571.548992    OdorSite      117.511265  20.000000             0      True   
241576.710976   InterSite      137.511265  22.016596             0       NaN   
241621.273984    OdorSite      159.527861  20.000000             0      True   
241625.544992   InterSite      179.527861  18.106378             0       NaN   
241628.881984    OdorSite      197.634239  20.000000             0      True   
241632.702976   InterSite      217.634239  20.500100            

In [ ]:
# Variables to plot
variables = ['is_choice', 'length', 'is_reward']
ylabel_map = {
    'is_choice': 'Total Choices',
    'length': 'Total Length (cm)',
    'is_reward': 'Total Rewards'
}
# === Setup ===
highlighted_mice = ['781898', '781896']
stages = ['A', 'B', 'C']
variable = 'is_reward'  # or 'length', 'is_reward'

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

# === Plot loop ===
for i, stage in enumerate(stages):
    ax = axes[i]
    reset = sum_df.loc[
        (sum_df.stage != 'shaping_stageC_distanceD_stopE_probB_equal') & 
        ((sum_df.simplified_stage == 'A')|
        ((sum_df.session_n <= 11)&(sum_df.simplified_stage == 'B'))|
        ((sum_df.session_n <= 19)&(sum_df.simplified_stage == 'C')))
    ].groupby(['mouse', 'session', 'session_n_stage', 'simplified_stage']).agg({
        'is_choice': 'sum',
        'length': 'sum', 
        'is_reward': 'sum'
    }).reset_index()
    
    # Background mice (greys)
    sns.lineplot(
        data=reset.loc[
            (reset.simplified_stage == stage) &
            (~reset.mouse.isin(highlighted_mice))
        ],
        x='session_n_stage',
        y=variable,
        hue='mouse',
        palette='Greys',
        alpha=0.5,
        style='simplified_stage',
        marker='.',
        legend=False,
        ax=ax
    )

    # Highlighted mice (oranges)
    line = sns.lineplot(
        data=reset.loc[
            (reset.simplified_stage == stage) &
            (reset.mouse.isin(highlighted_mice))
        ],
        x='session_n_stage',
        y=variable,
        hue='mouse',
        palette='Oranges',
        style='simplified_stage',
        marker='.',
        ax=ax
    )

    # Axis styling
    ax.set_title(f"Stage {stage}")
    if i == 0:
        handles, labels = line.get_legend_handles_labels()
        legend_handles = dict(zip(labels, handles))
    else:
        ax.set_ylabel('')
        ax.tick_params(left=False)           # no ticks
        
    ax.legend_.remove()
    ax.set_xlabel("Session Number")
    ax.set_ylabel(ylabel_map[variable])
    ax.tick_params(axis='x', rotation=45)
    sns.despine(ax=ax)

# One legend for mouse IDs (only for highlighted mice)
fig.legend(
    legend_handles.values(),
    legend_handles.keys(),
    title='Mouse',
    loc='center left',
    bbox_to_anchor=(0.9, 0.6),
    frameon=False
)
plt.tight_layout()
plt.subplots_adjust(right=0.85)


In [ ]:
stage = 'A'

In [ ]:
# Filter and aggregate
reset = sum_df.loc[
    sum_df.stage != 'shaping_stageC_distanceD_stopE_probB_equal'
].groupby(['mouse', 'session', 'session_n', 'session_n_stage', 'simplified_stage']).agg({
    'is_choice': 'sum',
    'length': 'sum', 
    'is_reward': 'sum'
}).reset_index()

# Variables to plot
variables = ['is_choice', 'length', 'is_reward']
ylabel_map = {
    'is_choice': 'Total Choices',
    'length': 'Total Length (cm)',
    'is_reward': 'Total Rewards'
}

# Create figure
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharex=True)

# Store legend handles and labels
legend_handles = []
legend_labels = []

for i, variable in enumerate(variables):
    ax = axes[i]

    # Plot background mice (no legend)
    sns.lineplot(
        data=reset.loc[
            (reset.simplified_stage == stage) &
            (~reset.mouse.isin(['781898', '781896']))
        ],
        x='session_n_stage',
        y=variable,
        hue='mouse',
        palette='Greys',
        alpha=0.5,
        style='simplified_stage',
        marker='.',
        legend=False,
        ax=ax
    )

    # Plot highlighted mice (capture legend only once)
    line = sns.lineplot(
        data=reset.loc[
            (reset.simplified_stage == stage) &
            (reset.mouse.isin(['781898', '781896'])) &
            (reset.session_n <= 10)
        ],
        x='session_n_stage',
        y=variable,
        hue='mouse',
        palette='Oranges',
        style='simplified_stage',
        marker='.',
        ax=ax
    )

    if i == 2:
        handles, labels = line.get_legend_handles_labels()
        legend_handles, legend_labels = handles, labels
    ax.legend_.remove()
    ax.set_ylabel(ylabel_map[variable])
    if i < 2:
        ax.set_xlabel('')
    else:
        ax.set_xlabel('Session Number')

    ax.tick_params(axis='x', rotation=45)
    sns.despine(ax=ax)

# Space for legend on right
plt.subplots_adjust(right=0.8)

# Add shared legend to the right
fig.legend(
    legend_handles, legend_labels,
    title='Mouse',
    loc='center left',
    bbox_to_anchor=(1.05, 0.5),
    frameon=False
)

plt.tight_layout()
